# LELA 60342  Research Methods in Computational Linguistics 2
### Week 2 CSF GPU Exercise

In this worksheet we are going to look at how to introduce the CSF into our work flow for the training and testing of model we saw last semester.

A reminder of the model:
Our dataset consists of just over 10,000 surnames, labelled with 18 different nationalities. The first tasks will be to learn a classifier that can accurately assign a nationality to previously unseen surnames. To do this we will use RNNs.

In [1]:
! wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/surnames_data.csv

--2026-02-07 11:57:25--  https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/surnames_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 179708 (175K) [text/plain]
Saving to: ‘surnames_data.csv’

surnames_data.csv   100%[===================>] 175.50K  --.-KB/s    in 0.01s   

2026-02-07 11:57:25 (11.5 MB/s) - ‘surnames_data.csv’ saved [179708/179708]



We read the data into a Pandas dataframe:

In [2]:
import pandas as pd
import torch
surnames_df=pd.read_csv("surnames_data.csv")
surnames_df

,nationality,surname
0,Arabic,Totah
1,Arabic,Abboud
2,Arabic,Fakhoury
3,Arabic,Srour
4,Arabic,Sayegh
...,...,...
10975,Vietnamese,Dinh
10976,Vietnamese,Phung
10977,Vietnamese,Quang
10978,Vietnamese,Vu


We then use hierachical indexing in Pandas to represent the data as sequences of separate characters

In [7]:
import torch
surnames_df=pd.read_csv("surnames_data.csv")

chars=[]
index_1=[]
index_2=[]
for i,row in surnames_df.iterrows():
    chars.extend(list(row.surname))
    index_1.extend([i]*len(row.surname))
    index_2.extend(range(len(row.surname)))

surnames_chars = pd.DataFrame(chars,index=[index_1,index_2])
surnames_chars.columns = ["chars"]
surnames_chars

chars
0     0     T
      1     o
      2     t
      3     a
      4     h
...       ...
10977 4     g
10978 0     V
      1     u
10979 0     H
      1     a

[73143 rows x 1 columns]

We can then use the Pandas function get_dummies to produce one-hot codings of the characters

In [8]:
surnames_oh=pd.get_dummies(surnames_chars.chars,dtype=int)
surnames_oh

'  -  /  1  :  A  B  C  D  E  ...  ö  ù  ú  ü  ą  ł  ń  Ś  Ż  ż
0     0  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      1  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      2  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      3  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      4  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
...     .. .. .. .. .. .. .. .. .. ..  ... .. .. .. .. .. .. .. .. .. ..
10977 4  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
10978 0  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      1  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
10979 0  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
      1  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0

[73143 rows x 84 columns]

And the nationalities

In [9]:
nationalities_oh=pd.get_dummies(surnames_df.nationality,dtype=int)
nationalities_oh

,Arabic,Chinese,Czech,Dutch,English,French,German,Greek,Irish,Italian,Japanese,Korean,Polish,Portuguese,Russian,Scottish,Spanish,Vietnamese
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10975,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
10976,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
10977,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
10978,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


We will then turn these into tensors for input to PyTorch and in particular to an LSTM layer. We want a tensor with the shape [Number_of_names, Number_of_characters_in name, Size_of_alphabet].

However the LSTM layer requires that all sequence be of the same length and so we pad our tensors by adding N tensors of zeros of the length of the one hot codings to the beginning of each name. So that the tensor actually has the form [Number_of_names, Number_of_characters_in_the_longest_name, Size_of_alphabet]

We do this using the function ZeroPad1d which takes as an argument a tuple with the following entries: padding_left, padding_right, padding_above, padding below.





In [10]:
from torch import nn
t=torch.ones([5,5,5])
print(t[0,:,:])
m = nn.ZeroPad1d((0,0,2,0))
print(m(t))

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
      

In [11]:
from torch import nn
# Find the length of the longest name in the data:
max_length=max([t[1] for t in surnames_oh.index])
# Make an array for the name tensors
X = [0] * (max(surnames_oh.index)[0]+1)
# Make an array for the label tensors
y = [0] * (max(nationalities_oh.index)+1)
# Iterate over index of the surnames one-hot data frame. The indices are tuples.
for ind in surnames_oh.index:
    # Make a tensor from subset of the dataframe for this name/index
    s=torch.from_numpy(surnames_oh.loc[ind[0]].values).to(dtype=torch.float)
    # Pad the tensor
    m = nn.ZeroPad1d((0,0,max_length-len(s),0))
    # Add tensors to arrays
    X[ind[0]] = m(s).cuda()
    y[ind[0]] = torch.from_numpy(nationalities_oh.loc[ind[0]].values).to(dtype=torch.float).cuda()
# Combine contents of arrays into a single tensor
X=torch.stack(X)
y=torch.stack(y)

In [12]:
X.shape

torch.Size([10980, 16, 84])

In [13]:
y.shape

torch.Size([10980, 18])

We can use torch.nn.Module to define our whole model

In [16]:
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
n_classes = 18

class SeqModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=84, hidden_size=42, num_layers=1, batch_first=True)
        self.linear = nn.Linear(42, n_classes)
    def forward(self, x):
        x, _ = self.rnn(x)
        # take only the last output
        x = x[:, -1, :]
        x = self.linear(x)
        return x

Now we have the model we can split the data then train and then test.

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

Instead of training our model we are going to import the weights.

In [25]:
model=SeqModel()
model.load_state_dict(torch.load("surname_model.pt"))
model.to("cuda:0")

SeqModel(
  (rnn): RNN(84, 42, batch_first=True)
  (linear): Linear(in_features=42, out_features=18, bias=True)
)

Our precision and recall are as follows:

In [26]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
y_test_pred=[np.argmax(x.cpu().detach().numpy()) for x in model(X_test)]
y_test_int=[np.argmax(x.cpu().detach().numpy()) for x in y_test]
precision_recall_fscore_support(y_test_int, y_test_pred, average='macro')


(0.3960275497503207, 0.39596274644711876, 0.38822480944968674, None)